# Multivariate Calculus -- Node 13

Partial derivatives, the multivariate chain rule, and directional derivatives, 
for $f : \mathbb{R}^n \to \mathbb{R}$.

Every neural-net loss is a function $\mathbb{R}^p \to \mathbb{R}$, and 
backpropagation is the chain rule applied through the computation graph. 
This notebook verifies the basic identities numerically with stdlib only.

## 1. Partial derivative

For $f : U \subset \mathbb{R}^n \to \mathbb{R}$ open and $x \in U$:
$$\frac{\partial f}{\partial x_i}(x) := \lim_{h \to 0} \frac{f(x + h e_i) - f(x)}{h}.$$

Centred differences approximate this with $O(h^2)$ error:
$$\frac{\partial f}{\partial x_i}(x) \approx \frac{f(x + h e_i) - f(x - h e_i)}{2 h}.$$

In [ ]:
import math

def f(x, y):
    return x * x * y + math.sin(x * y)

def partial_central(g, x, y, axis, h=1e-5):
    if axis == 0:
        return (g(x + h, y) - g(x - h, y)) / (2.0 * h)
    return (g(x, y + h) - g(x, y - h)) / (2.0 * h)

x0, y0 = 1.0, 2.0
dfdx_a = 2.0 * x0 * y0 + y0 * math.cos(x0 * y0)
dfdy_a = x0 * x0 + x0 * math.cos(x0 * y0)
dfdx_n = partial_central(f, x0, y0, 0)
dfdy_n = partial_central(f, x0, y0, 1)
print(f'df/dx analytic = {dfdx_a:.10f}, numeric = {dfdx_n:.10f}')
print(f'df/dy analytic = {dfdy_a:.10f}, numeric = {dfdy_n:.10f}')

## 2. Multivariate chain rule

If $f : \mathbb{R}^n \to \mathbb{R}$ is differentiable at $\gamma(t_0)$ and 
$\gamma : \mathbb{R} \to \mathbb{R}^n$ is differentiable at $t_0$, then
$$(f \circ \gamma)'(t_0) = \nabla f(\gamma(t_0)) \cdot \gamma'(t_0).$$

Setting $\gamma(t) = x + t v$ recovers the directional-derivative identity 
$D_v f(x) = \nabla f(x) \cdot v$.

In [ ]:
inv_sqrt2 = 1.0 / math.sqrt(2.0)
v = (inv_sqrt2, inv_sqrt2)

def directional_numeric(g, x, y, v, h=1e-5):
    return (g(x + h*v[0], y + h*v[1]) - g(x - h*v[0], y - h*v[1])) / (2.0 * h)

dv_formula = dfdx_a * v[0] + dfdy_a * v[1]
dv_numeric = directional_numeric(f, x0, y0, v)
print(f'D_v f  via grad . v   = {dv_formula:.10f}')
print(f'D_v f  via limit      = {dv_numeric:.10f}')
print(f'|err| = {abs(dv_formula - dv_numeric):.2e}')

# Chain rule along a circle: gamma(t) = (cos t, sin t), f(x,y) = x^2 + 3 y^2.
def g(x, y):
    return x * x + 3.0 * y * y

def gamma(t):
    return (math.cos(t), math.sin(t))

t0 = 0.7
x, y = gamma(t0)
dg_dx = 2.0 * x
dg_dy = 6.0 * y
gp = (-math.sin(t0), math.cos(t0))
chain = dg_dx * gp[0] + dg_dy * gp[1]
# Direct: f(gamma(t)) = cos^2 t + 3 sin^2 t = 1 + 2 sin^2 t; derivative = 4 sin t cos t = 2 sin(2t)
direct = 2.0 * math.sin(2.0 * t0)
print(f'\nChain rule check at t={t0}: chain={chain:.10f}, direct={direct:.10f}')

## 3. Exercises

1. **$C^1$ example.** Let $f(x,y) = e^x \cos y$. Compute $\nabla f(0, \pi/3)$ by hand. 
Then evaluate $D_v f$ at that point for $v = (3,4)/5$ and verify numerically.

2. **Partials exist but $f$ is not continuous.** Let
$$f(x,y) = \begin{cases} x^3 y / (x^4 + y^2) & (x,y) \neq (0,0) \\ 0 & (x,y) = (0,0). \end{cases}$$
Show $\partial_x f(0,0) = \partial_y f(0,0) = 0$ but $f$ is discontinuous at $(0,0)$ 
(consider $y = x^2$). Conclude $f$ is not (totally) differentiable at the origin.

3. **Chain rule.** Let $\gamma(t) = (e^t, t^2)$ and $f(x,y) = x y + \ln(1 + x^2)$. 
Compute $(f \circ \gamma)'(t)$ two ways and verify they match.